In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import pandas as pd



## loading the dataset


In [ ]:
file_path = Path("data") / "hacker_news.csv"

# If file doesn't exist in relative path, try absolute path
if not file_path.exists():
	# Adjust this path to where your file actually is
	file_path = Path("../data/hacker_news.csv")  # Try parent directory

df = pd.read_csv(file_path)
df_hn=df.copy()
df_hn.head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'data\\hacker_news.csv'

In [ ]:
df_hn.columns=['Id','Title','URL','Points','Comments','Author','Created_at']

In [ ]:
df_hn.info()


<class 'pandas.DataFrame'>
RangeIndex: 20100 entries, 0 to 20099
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Id          20100 non-null  int64
 1   Title       20100 non-null  str  
 2   URL         17660 non-null  str  
 3   Points      20100 non-null  int64
 4   Comments    20100 non-null  int64
 5   Author      20100 non-null  str  
 6   Created_at  20100 non-null  str  
dtypes: int64(3), str(4)
memory usage: 1.1 MB


In [ ]:
#statistics summary of the data
df_hn.describe()

,Id,Points,Comments
count,2.010000e+04,20100.000000,20100.000000
mean,1.131753e+07,50.296070,24.802289
std,6.964399e+05,107.107687,56.107340
min,1.017691e+07,1.000000,1.000000
25%,1.070176e+07,3.000000,1.000000
50%,1.128445e+07,9.000000,3.000000
75%,1.192607e+07,54.000000,21.000000
max,1.257898e+07,2553.000000,1733.000000


## Data Wrangling 

In [ ]:
#changing the data type of the 'created_at' column to datetime

df_hn['Created_at']=pd.to_datetime(df_hn['Created_at'])

In [ ]:
df_hn.info()

<class 'pandas.DataFrame'>
RangeIndex: 20100 entries, 0 to 20099
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Id          20100 non-null  int64         
 1   Title       20100 non-null  str           
 2   URL         17660 non-null  str           
 3   Points      20100 non-null  int64         
 4   Comments    20100 non-null  int64         
 5   Author      20100 non-null  str           
 6   Created_at  20100 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 1.1 MB


In [ ]:
#Clean data, replace null values with an object e.g (No url) for url column
print(df_hn.isnull().sum())
df_hn['URL']= df_hn['URL'].fillna('No URL provided')


Id               0
Title            0
URL           2440
Points           0
Comments         0
Author           0
Created_at       0
dtype: int64


## Extracting Ask HN and Show HN Posts


In [ ]:
#checking on posts with "Ask HN" and show "Show HN" in the title
ask_posts_text= 'Ask HN'.lower()
show_posts_text= 'Show HN'.lower()
show_posts= []
other_posts= []
ask_posts= []
hn_rows = df_hn.values.tolist()
for df_row in hn_rows:
    title= df_row[1]
    if title.lower().startswith(ask_posts_text):
        ask_posts.append(df_row)
    elif title.lower().startswith(show_posts_text):
        show_posts.append(df_row)
    else:
        other_posts.append(df_row)

print(f"Length of other posts: {len(other_posts)}")
print(f"Length of ask posts (Ask HN): {len(ask_posts)}")
print(f"Length of show posts (Show HN): {len(show_posts)}")


Length of other posts: 17194
Length of ask posts (Ask HN): 1744
Length of show posts (Show HN): 1162


## Calculating the Average Number of Comments for Ask HN and Show HN Posts

In [ ]:
total_ask_comments=0
total_show_comments=0
total_other_comments=0

for comments in ask_posts:
    total_ask_comments += comments[4]
avg_ask_comments= total_ask_comments/len(ask_posts)

for comments in other_posts:
    total_other_comments += comments[4] 
avg_other_comments= total_other_comments/len(other_posts)

for comments in show_posts:
    total_show_comments += comments[4]
avg_show_comments= total_show_comments/len(show_posts)

print(f"Average number of comments for Ask HN posts: {avg_ask_comments}")
print(f"Average number of comments for Show HN posts: {avg_show_comments}")
print(f"Average number of comments for other HN posts: {avg_other_comments}")

Average number of comments for Ask HN posts: 14.038417431192661
Average number of comments for Show HN posts: 10.31669535283993
Average number of comments for other HN posts: 26.8730371059672


## Finding the Number of Ask Posts and Comments by Hour Created

In [ ]:
import datetime as dt

In [ ]:
results_list= []
for post in ask_posts:  
    created_at= post[6]
    num_comments= post[4]
    results_list.append([created_at,num_comments])

In [ ]:
counts_by_hour= {}
comments_by_hour= {}
for result in results_list:
    created_at= result[0]
    num_comments= result[1]
    hour= created_at.hour
    if hour not in counts_by_hour:
        counts_by_hour[hour]= 1
        comments_by_hour[hour]= num_comments
    else:
        counts_by_hour[hour] += 1
        comments_by_hour[hour] += num_comments



{9: 45, 13: 85, 10: 59, 14: 107, 16: 108, 23: 68, 12: 73, 17: 100, 15: 116, 21: 109, 20: 80, 2: 58, 18: 109, 3: 54, 5: 46, 19: 110, 1: 60, 22: 71, 8: 48, 4: 47, 0: 55, 6: 44, 7: 34, 11: 58}
{9: 251, 13: 1253, 10: 793, 14: 1416, 16: 1814, 23: 543, 12: 687, 17: 1146, 15: 4477, 21: 1745, 20: 1722, 2: 1381, 18: 1439, 3: 421, 5: 464, 19: 1188, 1: 683, 22: 479, 8: 492, 4: 337, 0: 447, 6: 397, 7: 267, 11: 641}


In [ ]:
#Calculating the Average Number of Comments for Ask HN Posts by Hour
avrg_by_hour= []
for hour in counts_by_hour:
    avg_comments= comments_by_hour[hour]/ counts_by_hour[hour]
    avrg_by_hour.append([hour, avg_comments])



##  Sorting and Printing Values from a List of Lists

In [ ]:
swap_avg_by_hour = []

for row in avrg_by_hour:
    swap_avg_by_hour.append([row[1], row[0]])
    print(swap_avg_by_hour)
    sorted_swap = sorted(swap_avg_by_hour, reverse=True)



[[5.5777777777777775, 9]]
[[5.5777777777777775, 9], [14.741176470588234, 13]]
[[5.5777777777777775, 9], [14.741176470588234, 13], [13.440677966101696, 10]]
[[5.5777777777777775, 9], [14.741176470588234, 13], [13.440677966101696, 10], [13.233644859813085, 14]]
[[5.5777777777777775, 9], [14.741176470588234, 13], [13.440677966101696, 10], [13.233644859813085, 14], [16.796296296296298, 16]]
[[5.5777777777777775, 9], [14.741176470588234, 13], [13.440677966101696, 10], [13.233644859813085, 14], [16.796296296296298, 16], [7.985294117647059, 23]]
[[5.5777777777777775, 9], [14.741176470588234, 13], [13.440677966101696, 10], [13.233644859813085, 14], [16.796296296296298, 16], [7.985294117647059, 23], [9.41095890410959, 12]]
[[5.5777777777777775, 9], [14.741176470588234, 13], [13.440677966101696, 10], [13.233644859813085, 14], [16.796296296296298, 16], [7.985294117647059, 23], [9.41095890410959, 12], [11.46, 17]]
[[5.5777777777777775, 9], [14.741176470588234, 13], [13.440677966101696, 10], [13.23

In [ ]:
#top 5 hours for ask posts with the most comments
print("Top 5 Hours for Ask Posts Comments")
for avg, hr in sorted_swap[:5]:
    print(f"{hr}: {avg:.2f} average comments per post")


Top 5 Hours for Ask Posts Comments
15: 38.59 average comments per post
2: 23.81 average comments per post
20: 21.52 average comments per post
16: 16.80 average comments per post
21: 16.01 average comments per post


## Visulizations

### Plot 1: Average Comments — Ask HN vs Show HN vs Other


In [ ]:
categories = ["Ask HN", "Show HN", "Other"]
avg_comments = [avg_ask_comments, avg_show_comments, avg_other_comments]
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(categories, avg_comments, color=colors, width=0.5, edgecolor="white", linewidth=1.2)

# Add value labels on bars
for bar, val in zip(bars, avg_comments):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_title("Average Number of Comments by Post Type", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Post Type", fontsize=12)
ax.set_ylabel("Average Comments", fontsize=12)
ax.set_ylim(0, max(avg_comments) * 1.2)
sns.despine()
plt.tight_layout()
plt.show()


## Summary of Hacker News post's
### Inital Question: Compare "Ask Hacker News" and "Show Hacker News" posts to determine the following:

### 1. Do Ask HN or Show HN receive more comments on average?

**Answer**: On average, "ask posts" in the sample dataset receive's approximately 14 comment's on
average, while "show posts" receive's on average approximately 10 comments.

### 2. Do posts created at a certain time receive more comments on average?

**Answer**: Approx. 60% of comments are posted at 3:00pm, followed by 2:00am with an average of almost
24 comments.

## Overall conclusion:

### Of the posts data that "received comments":

1. The hour that receives the most comments per post on average is 15:00, with an average of 38.59
comments per post.
2. There appears to be an increase of approx. 60%, to the number of comments between the top two
hours for 'Ask HN' Comments.